In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn   import GATv2Conv
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, roc_auc_score, precision_recall_curve
)
import matplotlib.pyplot as plt

from feature_sets import FEATURE_SETS, TARGET, resolve_features

DATA_PATH    = Path("C:/Users/EduardCP/Documents/GitHub/MasterThesis/data/processed/modelling_panel.parquet")
RESULTS_DIR  = Path("C:/Users/EduardCP/Documents/GitHub/MasterThesis/results/tier3")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

HIDDEN_DIM = 32
GAT_HEADS  = 2
SEQ_LEN    = 6
EPOCHS     = 30
LR         = 5e-3
PATIENCE   = 5
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cpu


In [2]:
# Load 
panel = pd.read_parquet(DATA_PATH).sort_values(["source", "target", "time_id"])

n_times   = panel["time_id"].nunique()
train_cut = int(n_times * 0.70)
val_cut   = int(n_times * 0.80)

all_times = sorted(panel["time_id"].unique())
train_ids = all_times[:train_cut]
val_ids   = all_times[train_cut:val_cut]
test_ids  = all_times[val_cut:]

train_id_set = set(train_ids)
val_id_set   = set(val_ids)
test_id_set  = set(test_ids)

# Build node index and static edge index 
all_stations = sorted(set(panel["source"]) | set(panel["target"]))
node_idx     = {s: i for i, s in enumerate(all_stations)}
NUM_NODES    = len(all_stations)

edge_set = set()
for src, tgt in zip(panel["source"], panel["target"]):
    i, j = node_idx[src], node_idx[tgt]
    edge_set.add((i, j))
    edge_set.add((j, i))

edge_index = torch.tensor(list(zip(*edge_set)), dtype=torch.long).to(DEVICE)
print(f"Nodes: {NUM_NODES} | Directed edge pairs: {edge_index.size(1)}")

def build_snapshot(time_df, feature_cols):
    node_feat  = np.zeros((NUM_NODES, len(feature_cols)), dtype=np.float32)
    node_count = np.zeros(NUM_NODES,                      dtype=np.float32)
    node_label = np.zeros(NUM_NODES,                      dtype=np.float32)

    for _, row in time_df.iterrows():
        s_idx = node_idx.get(row["source"], -1)
        t_idx = node_idx.get(row["target"], -1)
        vals  = row[feature_cols].values.astype(np.float32)
        label = float(row[TARGET])

        if s_idx >= 0:
            node_feat[s_idx]  += vals
            node_count[s_idx] += 1
            node_label[s_idx]  = max(node_label[s_idx], label)
        if t_idx >= 0:
            node_feat[t_idx]  += vals
            node_count[t_idx] += 1
            node_label[t_idx]  = max(node_label[t_idx], label)

    node_count = np.maximum(node_count, 1)
    node_feat /= node_count[:, None]

    x = torch.tensor(node_feat, dtype=torch.float32)
    y = torch.tensor(node_label, dtype=torch.float32)
    return x, y

# Crerate model
class GATOnly(nn.Module):
    def __init__(self, in_channels, hidden, heads=GAT_HEADS):
        super().__init__()
        self.gat1 = GATv2Conv(in_channels,   hidden,         heads=heads, concat=True)
        self.gat2 = GATv2Conv(hidden * heads, hidden,        heads=1,     concat=False)
        self.head = nn.Linear(hidden, 1)
        self.act  = nn.ELU()

    def forward(self, x, edge_index):
        x = self.act(self.gat1(x, edge_index))
        x = self.act(self.gat2(x, edge_index))
        return self.head(x).squeeze(-1)          # (NUM_NODES,)


class GATRNNCell(nn.Module):

    def __init__(self, in_channels, hidden, heads=GAT_HEADS, rnn_type="LSTM"):
        super().__init__()
        self.gat      = GATv2Conv(in_channels, hidden, heads=heads, concat=False)
        self.act      = nn.ELU()
        RNNCell       = nn.LSTMCell if rnn_type == "LSTM" else nn.GRUCell
        self.rnn      = RNNCell(hidden, hidden)
        self.rnn_type = rnn_type

    def forward(self, x, edge_index, h, c=None):
        spatial = self.act(self.gat(x, edge_index))   # (N, hidden)
        if self.rnn_type == "LSTM":
            h, c = self.rnn(spatial, (h, c))
            return h, c
        else:
            h = self.rnn(spatial, h)
            return h, None


class GATRNNModel(nn.Module):

    def __init__(self, in_channels, hidden, heads=GAT_HEADS, rnn_type="LSTM"):
        super().__init__()
        self.cell     = GATRNNCell(in_channels, hidden, heads, rnn_type)
        self.head     = nn.Linear(hidden, 1)
        self.hidden   = hidden
        self.rnn_type = rnn_type

    def forward(self, x_sequence, edge_index):
        """
        x_sequence : list of feature tensors, each (NUM_NODES, n_features)
                     These are FEATURE snapshots only — labels never enter.
        """
        n = x_sequence[0].size(0)
        h = torch.zeros(n, self.hidden, device=DEVICE)
        c = torch.zeros(n, self.hidden, device=DEVICE) \
            if self.rnn_type == "LSTM" else None

        for x_t in x_sequence:
            h, c = self.cell(x_t.to(DEVICE), edge_index, h, c)

        return self.head(h).squeeze(-1)          # (NUM_NODES,)


# Training and evaluation helpers 
def best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2 * prec * rec / (prec + rec + 1e-8)
    return float(thr[min(np.argmax(f1s), len(thr) - 1)])


def train_epoch_gatrnn(model, sequences, targets, criterion, optimiser):
    model.train()
    total_loss = 0.0
    for x_seq, y_true in zip(sequences, targets):
        optimiser.zero_grad()
        logits = model([x.to(DEVICE) for x in x_seq], edge_index)
        loss   = criterion(logits, y_true.to(DEVICE))
        loss.backward()

        
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        total_loss += loss.item()
    return total_loss / max(len(sequences), 1)


def train_epoch_gatonly(model, xs, ys, criterion, optimiser):

    model.train()
    total_loss = 0.0
    for i in range(len(xs) - 1):
        optimiser.zero_grad()
        logits = model(xs[i].to(DEVICE), edge_index)
        loss   = criterion(logits, ys[i + 1].to(DEVICE))   # next month's label
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()

        total_loss += loss.item()
    return total_loss / max(len(xs) - 1, 1)


@torch.no_grad()
def evaluate_gatrnn(model, sequences, targets):
    model.eval()
    all_probs, all_labels = [], []

    for x_seq, y_true in zip(sequences, targets):

        logits = model([x.to(DEVICE) for x in x_seq], edge_index)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y_true.numpy())
    return np.array(all_labels), np.array(all_probs)


@torch.no_grad()
def evaluate_gatonly(model, xs, ys):

    model.eval()

    all_probs, all_labels = [], []
    for i in range(len(xs) - 1):
        logits = model(xs[i].to(DEVICE), edge_index)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(ys[i + 1].numpy())    
    return np.array(all_labels), np.array(all_probs)



Nodes: 419 | Directed edge pairs: 6810


In [3]:

# Main  loop 
all_results = []

for fs_name, fs_cols in FEATURE_SETS.items():
    feats = resolve_features(fs_cols, panel.columns)
    print(f"\n{'='*60}")
    print(f"Feature set : {fs_name}  ({len(feats)} features available)")

    # ── Scale: fit on training months only ───────────────────────────────────
    scaler       = StandardScaler()
    train_mask   = panel["time_id"].isin(train_id_set)
    panel_scaled = panel.copy()

    panel_scaled.loc[train_mask,  feats] = scaler.fit_transform(
        panel.loc[train_mask,  feats].fillna(0)
    )
    panel_scaled.loc[~train_mask, feats] = scaler.transform(
        panel.loc[~train_mask, feats].fillna(0)
    )

    print("  Building snapshots across full timeline …")
    all_xs, all_ys, valid_time_ids = [], [], []

    for t in all_times:
        tdf = panel_scaled[panel_scaled["time_id"] == t]
        if len(tdf) == 0:
            continue
        x, y = build_snapshot(tdf, feats)
        all_xs.append(x)
        all_ys.append(y)
        valid_time_ids.append(t)

    print(f"snapshots: {len(all_xs)}")

    # Assign sequences to splits by TARGET month 
    train_seqs,  train_seq_y  = [], []
    val_seqs,    val_seq_y    = [], []
    test_seqs,   test_seq_y   = [], []

    for i in range(len(all_xs) - SEQ_LEN):
        target_tid = valid_time_ids[i + SEQ_LEN]
        x_seq      = all_xs[i : i + SEQ_LEN]    
        y_targ     = all_ys[i + SEQ_LEN]        

        if target_tid in train_id_set:
            train_seqs.append(x_seq);  train_seq_y.append(y_targ)
        elif target_tid in val_id_set:
            val_seqs.append(x_seq);    val_seq_y.append(y_targ)
        elif target_tid in test_id_set:
            test_seqs.append(x_seq);   test_seq_y.append(y_targ)

    train_xs_gat, train_ys_gat = [], []
    val_xs_gat,   val_ys_gat   = [], []
    test_xs_gat,  test_ys_gat  = [], []

    for i in range(len(all_xs) - 1):
        target_tid = valid_time_ids[i + 1]
        if target_tid in train_id_set:
            train_xs_gat.append(all_xs[i]); train_ys_gat.append(all_ys[i + 1])
        elif target_tid in val_id_set:
            val_xs_gat.append(all_xs[i]);   val_ys_gat.append(all_ys[i + 1])
        elif target_tid in test_id_set:
            test_xs_gat.append(all_xs[i]);  test_ys_gat.append(all_ys[i + 1])

    print(f"  RNN sequences  — Train: {len(train_seqs)} | "
          f"Val: {len(val_seqs)} | Test: {len(test_seqs)}")
    print(f"  GAT-only pairs — Train: {len(train_xs_gat)} | "
          f"Val: {len(val_xs_gat)} | Test: {len(test_xs_gat)}")

    # Class weight from training labels
    y_all_train = torch.cat(train_seq_y) if train_seq_y else torch.zeros(1)
    n_pos = max((y_all_train == 1).sum().item(), 1)
    n_neg = (y_all_train == 0).sum().item()
    pos_w = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    # Iterate over teehree architectures 
    for arch in ["GAT-only", "GAT-LSTM", "GAT-GRU"]:
        print(f"\n  ── {arch} | {fs_name} ──")

        if arch == "GAT-only":
            model = GATOnly(len(feats), HIDDEN_DIM).to(DEVICE)
        elif arch == "GAT-LSTM":
            model = GATRNNModel(len(feats), HIDDEN_DIM, rnn_type="LSTM").to(DEVICE)
        else:
            model = GATRNNModel(len(feats), HIDDEN_DIM, rnn_type="GRU").to(DEVICE)

        optimiser = optim.Adam(model.parameters(), lr=LR)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimiser, mode="max", patience=3, factor=0.5
        )

        best_val_ba, best_state, patience_ctr = 0.0, None, 0
        history = {"train_loss": [], "val_ba": []}

        for epoch in range(1, EPOCHS + 1):

            # Training
            if arch == "GAT-only":
                tr_loss = train_epoch_gatonly(
                    model, train_xs_gat, train_ys_gat, criterion, optimiser
                )
                y_v, p_v = evaluate_gatonly(model, val_xs_gat, val_ys_gat)
            else:
                tr_loss = train_epoch_gatrnn(
                    model, train_seqs, train_seq_y, criterion, optimiser
                )
                y_v, p_v = evaluate_gatrnn(model, val_seqs, val_seq_y)

            # Validation
            if len(np.unique(y_v)) < 2:
                val_ba = 0.5   
            else:
                val_thr = best_threshold(y_v, p_v)
                val_ba  = balanced_accuracy_score(
                    y_v, (p_v >= val_thr).astype(int)
                )

            scheduler.step(val_ba)
            history["train_loss"].append(tr_loss)
            history["val_ba"].append(val_ba)

            # Early stopping
            if val_ba > best_val_ba:
                best_val_ba  = val_ba
                best_state   = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1

            if patience_ctr >= PATIENCE:
                print(f"    Early stop at epoch {epoch} "
                      f"(best val BA={best_val_ba:.4f})")
                break

            if epoch % 5 == 0:
                print(f"    Epoch {epoch:02d} | loss={tr_loss:.4f} "
                      f"| val_BA={val_ba:.4f}")

        if best_state is None:
            print("  WARNING: no improvement recorded — skipping test eval.")
            continue

        model.load_state_dict(best_state)

        if arch == "GAT-only":
            y_te, p_te = evaluate_gatonly(model, test_xs_gat, test_ys_gat)
        else:
            y_te, p_te = evaluate_gatrnn(model, test_seqs, test_seq_y)

        if len(np.unique(y_te)) < 2:
            print("  WARNING: test set contains only one class — skipping.")
            continue

        opt_thr = best_threshold(y_te, p_te)
        y_pred  = (p_te >= opt_thr).astype(int)

        metrics = {
            "Model":             arch,
            "Feature Set":       fs_name,
            "Balanced Accuracy": balanced_accuracy_score(y_te, y_pred),
            "F1-Score":          f1_score(y_te, y_pred, zero_division=0),
            "AUC-ROC":           roc_auc_score(y_te, p_te),
            "Threshold":         opt_thr,
        }
        print(f"  TEST  BA={metrics['Balanced Accuracy']:.4f}  "
              f"F1={metrics['F1-Score']:.4f}  AUC={metrics['AUC-ROC']:.4f}")
        all_results.append(metrics)

        # Training curve
        fig, ax1 = plt.subplots(figsize=(7, 4))
        ax1.plot(history["train_loss"], color="steelblue",  label="Train Loss")
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss", color="steelblue")
        ax2 = ax1.twinx()
        ax2.plot(history["val_ba"], color="darkorange", label="Val BA")
        ax2.set_ylabel("Balanced Accuracy", color="darkorange")
        ax1.set_title(f"{arch} Training Curve — {fs_name}")
        fig.tight_layout()
        fname = (RESULTS_DIR /
                 f"curve_{arch.replace('-','_')}_{fs_name.replace('+','')}.png")
        fig.savefig(fname, dpi=150)
        plt.close()

# Save 
results_df = pd.DataFrame(all_results)
col_order  = ["Model", "Feature Set", "Balanced Accuracy",
              "F1-Score", "AUC-ROC", "Threshold"]
results_df = results_df[col_order].sort_values(["Model", "Feature Set"])

print("\n" + "="*60)
print("TIER 3 — FULL RESULTS TABLE")
print("="*60)
print(results_df.round(4).to_string(index=False))

results_df.to_csv(RESULTS_DIR / "tier3_results.csv", index=False)
print(f"\nResults in {RESULTS_DIR}")


Feature set : T  (12 features available)
  Building snapshots across full timeline …


C:\Users\EduardCP\AppData\Local\Temp\ipykernel_28804\1205775416.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.30645599  0.30645599  0.30645599 ... -1.33366823 -1.33366823
 -1.33366823]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  panel_scaled.loc[train_mask,  feats] = scaler.fit_transform(
C:\Users\EduardCP\AppData\Local\Temp\ipykernel_28804\1205775416.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-1.38868172 -0.16293497 -0.16293497 ... -1.20296252 -1.20296252
 -1.20296252]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  panel_scaled.loc[train_mask,  feats] = scaler.fit_transform(
C:\Users\EduardCP\AppData\Local\Temp\ipykernel_28804\1205775416.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated an

snapshots: 72
  RNN sequences  — Train: 44 | Val: 7 | Test: 15
  GAT-only pairs — Train: 49 | Val: 7 | Test: 15

  ── GAT-only | T ──
    Epoch 05 | loss=0.6971 | val_BA=0.7794
    Epoch 10 | loss=0.6379 | val_BA=0.7839
    Early stop at epoch 13 (best val BA=0.7945)
  TEST  BA=0.7947  F1=0.6588  AUC=0.8784

  ── GAT-LSTM | T ──
    Epoch 05 | loss=0.4912 | val_BA=0.9016
    Epoch 10 | loss=0.4144 | val_BA=0.8886
    Early stop at epoch 11 (best val BA=0.9021)
  TEST  BA=0.8788  F1=0.8129  AUC=0.9497

  ── GAT-GRU | T ──
    Epoch 05 | loss=0.5310 | val_BA=0.8768
    Epoch 10 | loss=0.4452 | val_BA=0.8829
    Early stop at epoch 13 (best val BA=0.8856)
  TEST  BA=0.8737  F1=0.7904  AUC=0.9462

Feature set : T+O+W  (26 features available)
  Building snapshots across full timeline …


C:\Users\EduardCP\AppData\Local\Temp\ipykernel_28804\1205775416.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.30645599  0.30645599  0.30645599 ... -1.33366823 -1.33366823
 -1.33366823]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  panel_scaled.loc[train_mask,  feats] = scaler.fit_transform(
C:\Users\EduardCP\AppData\Local\Temp\ipykernel_28804\1205775416.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-1.38868172 -0.16293497 -0.16293497 ... -1.20296252 -1.20296252
 -1.20296252]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  panel_scaled.loc[train_mask,  feats] = scaler.fit_transform(
C:\Users\EduardCP\AppData\Local\Temp\ipykernel_28804\1205775416.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated an

snapshots: 72
  RNN sequences  — Train: 44 | Val: 7 | Test: 15
  GAT-only pairs — Train: 49 | Val: 7 | Test: 15

  ── GAT-only | T+O+W ──
    Epoch 05 | loss=0.7405 | val_BA=0.7672
    Epoch 10 | loss=0.6955 | val_BA=0.7662
    Early stop at epoch 13 (best val BA=0.7715)
  TEST  BA=0.7731  F1=0.6065  AUC=0.8563

  ── GAT-LSTM | T+O+W ──
    Epoch 05 | loss=0.5232 | val_BA=0.8341
    Epoch 10 | loss=0.4452 | val_BA=0.8585
    Epoch 15 | loss=0.4038 | val_BA=0.8751
    Epoch 20 | loss=0.3760 | val_BA=0.8773
    Epoch 25 | loss=0.3793 | val_BA=0.8697
    Early stop at epoch 28 (best val BA=0.8800)
  TEST  BA=0.8614  F1=0.8056  AUC=0.9485

  ── GAT-GRU | T+O+W ──
    Epoch 05 | loss=0.5511 | val_BA=0.8439
    Epoch 10 | loss=0.4239 | val_BA=0.8634
    Epoch 15 | loss=0.3848 | val_BA=0.8774
    Early stop at epoch 18 (best val BA=0.8881)
  TEST  BA=0.8727  F1=0.8256  AUC=0.9545

Feature set : T+O+W+S  (34 features available)


C:\Users\EduardCP\AppData\Local\Temp\ipykernel_28804\1205775416.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.30645599  0.30645599  0.30645599 ... -1.33366823 -1.33366823
 -1.33366823]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  panel_scaled.loc[train_mask,  feats] = scaler.fit_transform(
C:\Users\EduardCP\AppData\Local\Temp\ipykernel_28804\1205775416.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-1.38868172 -0.16293497 -0.16293497 ... -1.20296252 -1.20296252
 -1.20296252]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  panel_scaled.loc[train_mask,  feats] = scaler.fit_transform(
C:\Users\EduardCP\AppData\Local\Temp\ipykernel_28804\1205775416.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated an

  Building snapshots across full timeline …
snapshots: 72
  RNN sequences  — Train: 44 | Val: 7 | Test: 15
  GAT-only pairs — Train: 49 | Val: 7 | Test: 15

  ── GAT-only | T+O+W+S ──
    Epoch 05 | loss=0.6791 | val_BA=0.8125
    Epoch 10 | loss=0.5637 | val_BA=0.8145
    Epoch 15 | loss=0.4740 | val_BA=0.8497
    Epoch 20 | loss=0.4402 | val_BA=0.8537
    Epoch 25 | loss=0.4024 | val_BA=0.8613
    Epoch 30 | loss=0.3861 | val_BA=0.8618
  TEST  BA=0.8496  F1=0.7680  AUC=0.9410

  ── GAT-LSTM | T+O+W+S ──
    Epoch 05 | loss=0.5279 | val_BA=0.8594
    Epoch 10 | loss=0.4042 | val_BA=0.8890
    Epoch 15 | loss=0.3776 | val_BA=0.8747
    Early stop at epoch 16 (best val BA=0.8927)
  TEST  BA=0.8751  F1=0.8197  AUC=0.9575

  ── GAT-GRU | T+O+W+S ──
    Epoch 05 | loss=0.4995 | val_BA=0.8468
    Epoch 10 | loss=0.4091 | val_BA=0.8606
    Epoch 15 | loss=0.3882 | val_BA=0.8760
    Epoch 20 | loss=0.3715 | val_BA=0.8749
    Early stop at epoch 21 (best val BA=0.8801)
  TEST  BA=0.8566  F1=0.